# How 1-WL works inside Referential Alignment

This notebook explains, **step by step and assuming no prior knowledge**, how
Object Aligner uses **1-dimensional Weisfeiler–Leman (1-WL) color refinement**
to line up *id-bearing* objects between a `gold` and a `pred` structure — even
when the only thing telling two objects apart is *how they are wired into a
graph*, not any of their own visible fields.

We will:

1. Build a tiny `gold` / `pred` / `schema` example where every node looks
   identical and **only the graph structure can disambiguate them**.
2. Walk through the important methods in
   `src/object_aligner/_aligner_referential.py` (the plumbing that finds id
   scopes, builds the cost matrix, and runs the assignment) and
   `src/object_aligner/_aligner_wl.py` (the bridge that turns your data into an
   abstract graph and folds WL colors back into the cost).
3. Drive those internal methods **live**, printing their actual inputs and
   outputs, so you can see exactly what each one produces.

> Everything below calls the *real* internal methods. Nothing is faked.


## 0. The problem in one sentence

When you mark a field as an **id** (`idScope`) and other fields as
**references** to it (`ref`), the actual id *values* are meaningless — gold
might call someone `1` and pred might call the same someone `10`. To compare
references fairly, Object Aligner must first **discover a one-to-one mapping
(a bijection)** between gold ids and pred ids.

Usually the objects carry distinguishing attributes (a `name`, an `age`) and
the mapping falls out of ordinary property matching. But what if **several
objects are indistinguishable by their own fields**? Then their own properties
can't decide the mapping — yet their *position in the reference graph* often
can. That is exactly the job of 1-WL.


## 1. The example: a directed chain of identical people

Four people, **all named `"X"`** — so their own properties are useless for
telling them apart. The only structure is a directed chain of `relations`:

```
gold:  1 → 2 → 3 → 4        pred:  10 → 20 → 30 → 40
```

The two sides describe the *same* chain with different id values. The correct
bijection is `1↔10, 2↔20, 3↔30, 4↔40`, and it can only be found from the
*shape of the chain*.


In [1]:
from object_aligner import ObjectAligner

schema = {
    "type": "object",
    "properties": {
        "people": {
            "type": "array", "order": "align",
            "items": {
                "type": "object",
                "properties": {
                    "id":   {"type": "integer", "idScope": "person"},  # <- id definer
                    "name": {"type": "string"},
                },
            },
        },
        "relations": {
            "type": "array", "order": "align",
            "items": {
                "type": "object",
                "properties": {
                    "source": {"type": "integer", "ref": "person"},    # <- reference
                    "target": {"type": "integer", "ref": "person"},    # <- reference
                },
            },
        },
    },
}

gold = {
    "people": [{"id": i, "name": "X"} for i in (1, 2, 3, 4)],
    "relations": [
        {"source": 1, "target": 2},
        {"source": 2, "target": 3},
        {"source": 3, "target": 4},
    ],
}
pred = {
    # Same chain, but the people are listed in a *scrambled* order on purpose:
    # a model emits entities in any order, and we want to prove WL recovers the
    # mapping from structure, not from list position.
    "people": [{"id": i, "name": "X"} for i in (40, 10, 30, 20)],
    "relations": [
        {"source": 10, "target": 20},
        {"source": 20, "target": 30},
        {"source": 30, "target": 40},
    ],
}

aligner = ObjectAligner(schema)  # id_disambiguation="wl" is the default
print("score with WL (default):", aligner.metric(gold, pred))
print("score with WL disabled :", ObjectAligner(schema, id_disambiguation="none").metric(gold, pred))


score with WL (default): {'score': 1.0}
score with WL disabled : {'score': 0.625}


**Read that result.** With WL on, the score is a perfect `1.0`: the chain
is recognized as identical. With `id_disambiguation="none"`, the aligner falls
back to an *arbitrary* tie-break among the four identical people; because we
listed pred's people in a scrambled order, that arbitrary choice mismatches
most references and the score drops to `0.625`. That gap is the entire value of
WL — and the rest of the notebook explains how the `1.0` is earned.


## 2. The big picture: who calls whom

When you call `aligner.metric(gold, pred)` (or `align`), referential setup runs
*before* the normal recursive scoring. Here is the call chain, with the file
each piece lives in:

```
align()  /  metric()                                   [object_aligner.py]
└─ _align_with_ctx()
   ├─ _validate_referential(gold)        gold id sets   [_aligner_referential.py]
   ├─ _collect_pred_ids(pred)            pred id sets    [_aligner_referential.py]
   ├─ _derive_id_mappings()              THE BIJECTION   [_aligner_referential.py]
   │  └─ _derive_single_scope()          per scope
   │     ├─ _align_helper(...)           property cost matrix   [_aligner_core.py]
   │     ├─ _build_ref_graph()           data  ->  abstract graph   [_aligner_wl.py]
   │     ├─ wl_tokens()                  graph -> per-side colors   [_wl.py]
   │     └─ _apply_wl()                  fold colors into cost      [_aligner_wl.py]
   │     └─ linear_sum_assignment()      Hungarian -> mapping       [scipy]
   └─ _align_helper(gold, pred, ...)     normal scoring; refs use the mapping
```

We will follow this top to bottom. First the *plumbing*
(`_aligner_referential.py`), then the *graph + WL* (`_aligner_wl.py`,
`_wl.py`), then how the colors get folded back in and the mapping is read off.


## 3. `_collect_id_scopes` — finding the ids and refs in your schema

This runs **once, at construction time** (inside `__init__`). It walks the
schema looking for the custom keywords `idScope` and `ref`, and records, for
each scope:

- **`definer_schema_path`** — the schema path to the id field itself
  (`people[*].id`).
- **`definer_array_path`** — the schema path to the *array* the definers live
  in (`people`). The ids must live in an array, because the definers form an
  *alignable list*.
- **`ref_paths`** — every place a `ref` into this scope appears
  (`relations[*].source`, `relations[*].target`).

Paths are tuples of *edges*: `("properties", "people")` descends into a dict
key, `("items",)` iterates an array. We can read all of this straight off the
constructed aligner.


In [2]:
scope = aligner._id_scopes["person"]   # an _IdScope dataclass
print("scope name           :", scope.scope)
print("primitive type       :", scope.primitive_type)
print("definer_array_path   :", scope.definer_array_path)
print("definer_schema_path  :", scope.definer_schema_path)
print("ref_paths            :")
for rp in scope.ref_paths:
    print("    ", rp)
print("degraded (in a cycle):", scope.degraded)


scope name           : person
primitive type       : integer
definer_array_path   : (('properties', 'people'), ('items',))
definer_schema_path  : (('properties', 'people'), ('items',), ('properties', 'id'))
ref_paths            :
     (('properties', 'relations'), ('items',), ('properties', 'source'))
     (('properties', 'relations'), ('items',), ('properties', 'target'))
degraded (in a cycle): False


So the aligner now *knows*: "the ids live at `people[*].id`, and they are
referenced from `relations[*].source` and `relations[*].target`." Notice the
id values (`1,2,3,4`) appear **nowhere** here — this is purely about *where in
the shape* ids and refs sit.


## 4. `_toposort_scopes` — what order to resolve scopes in

If scope **A**'s objects contain references to scope **B**, then B's mapping
must be solved *first* (A can then use B's already-known mapping as extra
evidence). `_toposort_scopes` topologically sorts the scopes for this reason,
and detects dependency **cycles** (which it warns about and degrades to
property-only alignment).

We only have one scope, so the order is trivial — but here is the result the
aligner stored:


In [3]:
print("scope resolution order:", aligner._scope_order)

scope resolution order: ('person',)


With multiple scopes (say `person` referenced by `pet.owner`), this would
be the list that guarantees `person` is solved before `pet`. The WL label
builder later *exploits* this: it can fold an already-resolved higher scope's
mapping into the graph labels (see `_carrier_label`).


## 5. `_walk_data` — turning a schema path into actual values

A schema path like `(("properties","people"), ("items",))` is a *recipe*.
`_walk_data(data, path)` follows that recipe through a concrete object and
yields every `(value, data_path)` it reaches. `("items",)` fans out over all
list elements, so one path can yield many values.

This is the workhorse used everywhere below to go from "where the schema says
ids live" to "the actual id values in this object."


In [4]:
# Every person object reached by the definer array path:
for value, data_path in aligner._walk_data(gold, scope.definer_array_path):
    print("at", data_path, "->", value)

print()
# Every source/target ref *value* in gold:
for rp in scope.ref_paths:
    vals = [v for v, _ in aligner._walk_data(gold, rp)]
    print(rp[-1], "values:", vals)


at ('people', 0) -> {'id': 1, 'name': 'X'}
at ('people', 1) -> {'id': 2, 'name': 'X'}
at ('people', 2) -> {'id': 3, 'name': 'X'}
at ('people', 3) -> {'id': 4, 'name': 'X'}

('properties', 'source') values: [1, 2, 3]
('properties', 'target') values: [2, 3, 4]


## 6. `_validate_referential` and `_collect_pred_ids` — the id sets

Before any matching, the aligner gathers the set of ids on each side.

- **`_validate_referential(gold)`** is *strict*: duplicate gold ids or a gold
  ref pointing at a non-existent id raise a `ValidationError`. Gold is your
  ground truth, so it must be internally consistent. It returns the gold id set
  per scope.
- **`_collect_pred_ids(pred)`** is *tolerant*: pred is a model's output and may
  be malformed, so duplicates are dropped (first wins) and nothing raises.

These sets get stored on the per-call context (`ctx`). Let's build a `ctx` the
same way `align()` does and populate them.


In [5]:
from object_aligner._matchtypes import _AlignContext

ctx = _AlignContext()
ctx.gold_ids = aligner._validate_referential(gold)
ctx.pred_ids = aligner._collect_pred_ids(pred)
print("gold ids:", ctx.gold_ids)
print("pred ids:", ctx.pred_ids)


gold ids: {'person': {1, 2, 3, 4}}
pred ids: {'person': {40, 10, 20, 30}}


## 7. `_derive_single_scope` — the property cost matrix (and why it's useless here)

This is the heart of `_derive_id_mappings`. For one scope it:

1. Lists the gold definer objects and the pred definer objects.
2. Builds an $n \times m$ **cost matrix** where `cost[i][j]` is the similarity
   of gold person *i* to pred person *j*, scored by **their ordinary
   properties** — `_align_helper` is called on the two person objects.

**The masking trick.** While scoring person *i* vs person *j*, any `ref` into
*this same scope* is forced to score `1.0` (via `ctx.mask_scope`). Why? Because
the mapping for this scope doesn't exist yet — using refs into it would be
circular ("I'll decide the mapping using the mapping I'm trying to decide").
So the cost matrix is built from **non-self-referential properties only**.

Let's build it by hand exactly as `_derive_single_scope` does.


In [6]:
import numpy as np

ctx.mask_scope = "person"   # mask refs into the scope we're currently solving
item_schema = aligner._get_schema_node(aligner.schema, scope.definer_array_path)

gold_items = list(aligner._walk_data(gold, scope.definer_array_path))
pred_items = list(aligner._walk_data(pred, scope.definer_array_path))
n, m = len(gold_items), len(pred_items)

suffix = scope.definer_schema_path[len(scope.definer_array_path):]  # path from item -> id
def extract_id(item):
    for val, _ in aligner._walk_data(item, suffix):
        return val
gold_id_list = [extract_id(it) for it, _ in gold_items]
pred_id_list = [extract_id(it) for it, _ in pred_items]

d = max(n, m)
cost = np.zeros((d, d))
for i in range(n):
    for j in range(m):
        cost[i][j] = aligner._align_helper(gold_items[i][0], pred_items[j][0], item_schema, ctx)["match"].score

print("gold ids:", gold_id_list)
print("pred ids:", pred_id_list)
print("property cost matrix (rows = gold people, cols = pred people):")
print(cost)


gold ids: [1, 2, 3, 4]
pred ids: [40, 10, 30, 20]
property cost matrix (rows = gold people, cols = pred people):
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]


**Look at that matrix: every entry is `1.0`.** Every person is named `"X"`
and the id field is masked, so *by properties alone* every gold person matches
every pred person equally well. The Hungarian algorithm would have $4! = 24$
equally-good assignments to choose from and would pick one **arbitrarily**.
This is precisely the ambiguity WL exists to resolve.


## 8. `_build_ref_graph` — turning your data into an abstract graph

Now we leave `_aligner_referential.py` for `_aligner_wl.py`. WL doesn't know
anything about JSON or schemas — it works on a plain **directed, labeled
graph**. `_build_ref_graph` is the translator. It produces a `RefGraph` with:

- **`vertices`** — one per definer id (here: `1,2,3,4`). (Plus synthetic "hub"
  vertices for symmetric/k-ary relations; none here.)
- **`incidences`** — one `_RefEdge(src, dst, role, label)` per reference
  relation, connecting the referenced ids.

The clever part is deciding *what each edge connects*. That's `_carrier_path`
and `_emit_incidences`, covered next — but first let's just build the graph and
look at it.


In [7]:
gold_graph = aligner._build_ref_graph(gold, scope, ctx, is_gold=True)
pred_graph = aligner._build_ref_graph(pred, scope, ctx, is_gold=False)

print("GOLD vertices:", dict(gold_graph.vertices))
print("GOLD edges:")
for e in gold_graph.incidences:
    print(f"    {e.src} -> {e.dst}   role={e.role}   label={e.label}")
print()
print("PRED vertices:", dict(pred_graph.vertices))
print("PRED edges:")
for e in pred_graph.incidences:
    print(f"    {e.src} -> {e.dst}   role={e.role}   label={e.label}")


GOLD vertices: {1: (), 2: (), 3: (), 4: ()}
GOLD edges:
    1 -> 2   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()
    2 -> 3   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()
    3 -> 4   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()

PRED vertices: {40: (), 10: (), 30: (), 20: ()}
PRED edges:
    10 -> 20   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()
    20 -> 30   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()
    30 -> 40   role=('edge', (('properties', 'source'),), (('properties', 'target'),))   label=()


Each `relations` entry `{source: a, target: b}` became a directed edge
`a → b`. Gold is the chain `1→2→3→4`; pred is `10→20→30→40`. **Same shape,
different labels on the vertices.** WL will exploit the shape.

Note `label=()` and `vertices` map to `()`: under the default `"tie_break"`
mode WL uses **structure only** — no node/edge attributes are seeded. (Under
`"blend"` mode the vertices and labels would carry the objects' exact scalars.)


### 8a. `_carrier_path` — *what* owns each reference

A reference doesn't float free; it sits inside some object — its **carrier**.
`_carrier_path` finds the smallest enclosing *object-that-is-an-array-item* for
a ref site. Here both `source` and `target` live in the same `relations[*]`
item, so they share one carrier — which is what lets `_emit_incidences` pair
them into a single directed edge `source → target` instead of two stray tags.


In [8]:
for rp in scope.ref_paths:
    cp = aligner._carrier_path(rp)
    print(f"ref {rp[-1]!s:30}  carrier = {cp}")


ref ('properties', 'source')        carrier = (('properties', 'relations'), ('items',))
ref ('properties', 'target')        carrier = (('properties', 'relations'), ('items',))


Both refs report the **same** carrier path (`relations[*]`). That shared
carrier is the signal "these two ids participate in *one* relation together."
For a `members: [ref, ref, ...]` array the carrier would instead be the *group*
object, so all co-members form one k-ary relation (a star), not isolated tags.


### 8b. `_emit_incidences` — *how* a carrier becomes edges

Given the endpoints a carrier collects, `_emit_incidences` chooses one of three
shapes, and this choice is what keeps everything inside the power of 1-WL:

| # endpoints | distinct roles | becomes | example |
|---|---|---|---|
| 1 | — | a **unary self-tag** (`src == dst`) | a lone `ref` field |
| 2 | 2 | a **directed edge** `src → dst` | `{source, target}` |
| otherwise | — | a **star to a fresh hub vertex** | symmetric `members: [...]` |

The directed-edge case is why our `{source, target}` carrier produces exactly
the `a → b` edges we saw. The role tuple records *which* ref field was which
(`source` vs `target`), so direction is preserved.


### 8c. `_carrier_label` and `_exact_scalars` — baking in hard evidence

Edges and vertices can carry a **label**: a sorted tuple of the carrier's own
*exactly-comparable* scalars (strings, ints, bools, enums — but **not** floats,
which would split structurally-identical edges over rounding noise), plus any
references to an **already-resolved higher scope** (mapped into pred space on
the gold side so both sides match).

In our example there are no such scalars on a `relations` item, so the labels
are empty `()`. If each relation had a `"type"` (e.g. `"parent_of"` vs
`"child_of"`), that string would appear in the label and WL would refuse to
match a `parent_of` edge against a `child_of` edge. Let's demonstrate by
calling it directly on the first relation object:


In [9]:
rel0 = gold["relations"][0]            # {"source": 1, "target": 2}
carrier_path = aligner._carrier_path(scope.ref_paths[0])
label = aligner._carrier_label(rel0, carrier_path, scope, ctx, is_gold=True)
print("exact scalars on this relation:", aligner._exact_scalars(rel0, aligner._get_schema_node(aligner.schema, carrier_path)))
print("carrier label:", label, " (empty -> structure-only here)")


exact scalars on this relation: ()
carrier label: ()  (empty -> structure-only here)


## 9. `wl_tokens` — the actual color refinement

Now the payoff. `wl_tokens` (in `_wl.py`) takes the two graphs and assigns each
vertex an integer **color** such that two vertices get the same color **iff
1-WL cannot tell them apart**. Two things make it special:

1. **Joint refinement over the disjoint union.** Gold and pred graphs are
   refined *together*, with a single shared `signature → token` dictionary each
   round. So if a gold vertex and a pred vertex have the identical local
   structure, they receive the **same integer** — and that is what makes
   per-side colors comparable across sides **without ever consulting a
   cross-side mapping** (no chicken-and-egg).
2. **The 1-WL update rule.** Every round, each vertex's new color is a hash of
   its old color plus the *sorted multiset* of its neighbors' colors:

$$c_{t+1}(v) = \mathrm{relabel}\Big(c_t(v),\ \{\!\!\{\,(\text{dir},\,\text{role},\,\text{label},\,c_t(u)) : u \in N(v)\,\}\!\!\}\Big)$$

Refinement repeats until the number of distinct colors stops growing (a stable
partition). Here is the real result:


In [10]:
from object_aligner._wl import wl_tokens

gold_tokens, pred_tokens = wl_tokens(gold_graph, pred_graph, mode="tie_break")
print("gold tokens:", gold_tokens)
print("pred tokens:", pred_tokens)


gold tokens: {1: 3, 2: 1, 3: 0, 4: 2}
pred tokens: {40: 2, 10: 3, 30: 0, 20: 1}


**Every vertex got a distinct color, and the colors match across sides:**
`1↔10`, `2↔20`, `3↔30`, `4↔40` all share a color. The chain is fully resolved.
But where did those numbers come from? Let's open up the loop and watch it
round by round.


### 9a. Watching refinement round by round

The cell below is a **faithful re-implementation** of the loop inside
`wl_tokens` (same disjoint-union adjacency, same shared relabeling), instrumented
to print the colors after each round. This is *exactly* what the library does
internally; we only added `print`s.


In [11]:
# Build the disjoint-union adjacency, keyed by (side, vertex_id) — side 0 = gold, 1 = pred.
graphs = {0: gold_graph, 1: pred_graph}
adjacency = {}
for side, graph in graphs.items():
    for vid in graph.vertices:
        adjacency[(side, vid)] = []
for side, graph in graphs.items():
    for e in graph.incidences:
        if (side, e.src) in adjacency:
            adjacency[(side, e.src)].append(("out", e.role, e.label, (side, e.dst)))
        if (side, e.dst) in adjacency:
            adjacency[(side, e.dst)].append(("in", e.role, e.label, (side, e.src)))

keys = list(adjacency)

def relabel(signatures):
    # Distinct signatures -> small ints, ordered by repr (deterministic, hash-seed-independent).
    distinct = sorted({signatures[k] for k in keys}, key=repr)
    token = {sig: i for i, sig in enumerate(distinct)}
    return {k: token[signatures[k]] for k in keys}

def show(color, title):
    g = {vid: color[(0, vid)] for vid in gold_graph.vertices}
    p = {vid: color[(1, vid)] for vid in pred_graph.vertices}
    nparts = len({color[k] for k in keys})
    print(f"{title:9}  gold={g}  pred={p}   ({nparts} distinct colors)")

# Round 0: everyone starts with the same color (tie_break seeds a constant).
color = relabel({k: ("c0", ()) for k in keys})
show(color, "round 0")

prev_parts = len({color[k] for k in keys})
for r in range(1, len(keys) + 1):
    signatures = {}
    for k in keys:
        neighborhood = sorted(
            ((d, role, label, color[nb]) for (d, role, label, nb) in adjacency[k]),
            key=repr,
        )
        signatures[k] = (color[k], tuple(neighborhood))
    color = relabel(signatures)
    show(color, f"round {r}")
    parts = len({color[k] for k in keys})
    if parts == prev_parts:
        print("           -> partition stable; stop.")
        break
    prev_parts = parts


round 0    gold={1: 0, 2: 0, 3: 0, 4: 0}  pred={40: 0, 10: 0, 30: 0, 20: 0}   (1 distinct colors)
round 1    gold={1: 2, 2: 0, 3: 0, 4: 1}  pred={40: 1, 10: 2, 30: 0, 20: 0}   (3 distinct colors)
round 2    gold={1: 3, 2: 1, 3: 0, 4: 2}  pred={40: 2, 10: 3, 30: 0, 20: 1}   (4 distinct colors)
round 3    gold={1: 3, 2: 1, 3: 0, 4: 2}  pred={40: 2, 10: 3, 30: 0, 20: 1}   (4 distinct colors)
           -> partition stable; stop.


**Read the rounds like a story:**

- **Round 0** — everyone is color `0`. Total ignorance.
- **Round 1** — the *endpoints* reveal themselves. Node `1` has an outgoing
  edge but no incoming one; node `4` has incoming but no outgoing. They split
  off into their own colors. Nodes `2` and `3` both have exactly one-in and
  one-out, so they still share a color.
- **Round 2** — now `2` and `3` look at *their neighbors' round-1 colors*.
  Node `2`'s predecessor is the "head" `1`; node `3`'s predecessor is a middle
  node. That difference splits them. All four colors are now distinct.
- **Round 3** — nothing new splits, so the partition is stable and we stop.

And crucially, because gold and pred were refined **together**, the gold chain
and the pred chain received the *same* color sequence — that is what lets us
read the bijection straight off matching colors.


## 10. `_apply_wl` — folding colors back into the cost matrix

WL gave us colors; now we turn "same color" into a nudge on the cost matrix.
First the aligner builds an **agreement matrix** `w` where `w[i][j] = 1` iff
gold person *i* and pred person *j* have the same WL color. Then `_apply_wl`
combines it with the property `cost` in one of two ways:

- **`"tie_break"` (default)** — add a *tiny* `eps * w`, with `eps` chosen
  strictly smaller than the smallest gap between distinct property scores. This
  **only breaks exact ties**: pairs already separated by their properties keep
  their ranking; structure decides only when properties are silent.
- **`"blend"`** — mix them as $(1-\lambda)\cdot\text{cost} + \lambda\cdot w$,
  letting structure outweigh properties to degree $\lambda$.

Let's build `w` and apply the default tie-break.


In [12]:
w = np.zeros((d, d))
for i in range(n):
    gtok = gold_tokens.get(gold_id_list[i])
    for j in range(m):
        if gtok is not None and pred_tokens.get(pred_id_list[j]) == gtok:
            w[i][j] = 1.0

print("WL agreement matrix w (1 = same color):")
print(w)

score_matrix = aligner._apply_wl(cost, w, n, m)
print("\nscore matrix after tie-break (cost + eps*w):")
print(score_matrix)


WL agreement matrix w (1 = same color):
[[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]

score matrix after tie-break (cost + eps*w):
[[1.         1.05882353 1.         1.        ]
 [1.         1.         1.         1.05882353]
 [1.         1.         1.05882353 1.        ]
 [1.05882353 1.         1.         1.        ]]


The property `cost` was a flat field of `1.0`s — totally tied. After the
tie-break, the diagonal `1↔10, 2↔20, 3↔30, 4↔40` is now *infinitesimally*
higher than every off-diagonal cell. The numbers barely changed (`eps` is tiny
on purpose), but the ordering is no longer ambiguous: there is now a single
best assignment.


## 11. The Hungarian step — reading off the bijection

Finally `_derive_single_scope` runs `scipy.optimize.linear_sum_assignment` on
`-score_matrix` (negated because the solver minimizes, and we want to maximize
similarity). The chosen `(row, col)` pairs are translated back into a
`gold_id -> pred_id` mapping.


In [13]:
from scipy.optimize import linear_sum_assignment

row_ind, col_ind = linear_sum_assignment(-score_matrix)
mapping = {}
matched_pred = set()
for ri, ci in zip(row_ind, col_ind):
    if ri < n and ci < m:
        g_id, p_id = gold_id_list[ri], pred_id_list[ci]
        if p_id is not None and p_id not in matched_pred:
            mapping[g_id] = p_id
            matched_pred.add(p_id)
        else:
            mapping[g_id] = None
print("derived bijection (gold id -> pred id):", mapping)


derived bijection (gold id -> pred id): {1: 10, 2: 20, 3: 30, 4: 40}


And the same thing through the **public path**, to prove we reproduced the
real internals exactly:


In [14]:
match, real_ctx = aligner._align_with_ctx(gold, pred)
print("mapping the library actually derived:", real_ctx.current_mappings["person"])
print("final score:", aligner.metric(gold, pred))


mapping the library actually derived: {1: 10, 2: 20, 3: 30, 4: 40}
final score: {'score': 1.0}


With the bijection in hand, normal scoring resumes: when `_align_helper`
hits a `ref` field, it looks the gold id up in the mapping and checks whether
pred points at the mapped id (see the `ref` short-circuit in
`_aligner_core.py`). Every reference in the chain now matches, so the score is
a perfect `1.0`.


## 12. When WL *can't* help — and shouldn't

1-WL is powerful but not omniscient. Two situations leave **residual
ambiguity**, which is correct behavior — there is genuinely no information to
choose:

1. **Genuine automorphisms.** If swapping two nodes is itself a perfect
   symmetry of the graph, no algorithm can prefer one pairing. Below, *both*
   identical people belong to the group, so swapping them changes nothing.
2. **1-WL blind spots.** Famously, one 6-cycle vs two disjoint 3-cycles: every
   vertex is 2-regular, so 1-WL paints them all the same color even though the
   graphs differ. (Higher-order $k$-WL would separate them; the library uses
   1-WL by design.)

`warn_on_ambiguous_mapping=True` fires **only** on this residual case. Watch:


In [15]:
import warnings

twin_schema = {
    "type": "object",
    "properties": {
        "people": {"type": "array", "order": "align", "items": {
            "type": "object", "properties": {
                "id": {"type": "integer", "idScope": "person"},
                "name": {"type": "string"},
            }}},
        "groups": {"type": "array", "order": "align", "items": {
            "type": "object", "properties": {
                "name": {"type": "string"},
                "members": {"type": "array", "order": "align",
                            "items": {"type": "integer", "ref": "person"}},
            }}},
    },
}
g_auto = {"people": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Alice"}],
          "groups": [{"name": "g", "members": [1, 2]}]}      # BOTH Alices in the group
p_auto = {"people": [{"id": 10, "name": "Alice"}, {"id": 20, "name": "Alice"}],
          "groups": [{"name": "g", "members": [10, 20]}]}

twin_aligner = ObjectAligner(twin_schema, warn_on_ambiguous_mapping=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("score:", twin_aligner.metric(g_auto, p_auto))
for wmsg in caught:
    print("WARNING:", wmsg.message)


score: {'score': 1.0}


The pairing is ambiguous, but **either choice gives the same score**, so
the residual ambiguity is harmless — the warning is just transparency. Contrast
with our chain example, where exactly *one* Alice being in the group (an
asymmetry) would let WL pin the mapping with no warning at all.


## 13. Recap — the whole pipeline in one breath

1. **`_collect_id_scopes`** (build time) finds where ids and refs live in the
   schema.
2. **`_toposort_scopes`** orders scopes so referenced scopes resolve first.
3. **`_validate_referential` / `_collect_pred_ids`** gather the id sets.
4. **`_derive_single_scope`** builds a property **cost matrix** with self-scope
   refs *masked* (no bootstrapping). When objects are property-twins this
   matrix is tied.
5. **`_build_ref_graph`** (`_carrier_path`, `_emit_incidences`,
   `_carrier_label`) translates the data into an abstract directed labeled
   graph — *per side, independently*.
6. **`wl_tokens`** refines both graphs **jointly over their disjoint union**,
   producing per-vertex colors that are **comparable across sides** with no
   cross-side mapping needed.
7. **`_apply_wl`** folds "same color" into the cost — a hair-thin tie-break by
   default, a weighted blend optionally.
8. **`linear_sum_assignment`** reads the bijection off the nudged cost.
9. Normal scoring resumes; `ref` fields compare *through* the discovered
   mapping.

The one idea to keep: **when an object's own fields can't tell it apart from
its twins, its position in the reference graph usually can — and 1-WL is the
machinery that turns "position in the graph" into a comparable color on each
side.**
